# 1. Classification

## Question 1

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, confusion_matrix, ConfusionMatrixDisplay,
    f1_score, classification_report, mean_squared_error, r2_score
)
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from statistics import mean, stdev
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import PolynomialFeatures

import warnings
warnings.filterwarnings("ignore")

In [ ]:
data = pd.read_csv("data.csv", delimiter=";")
data.head()


In [ ]:
data['Target'] = data['Target'].replace({'Dropout': 0, 'Graduate': 1, 'Enrolled': 1})
data_all = data.copy()
data_target = data['Target']
data = data.drop('Target', axis=1)

## Question 2

In [ ]:
# on normalise les valeurs de chaque attribut (tous sont de type numérique)
scaler = StandardScaler()
scaled_data = scaler.fit_transform(data)
scaled_data = pd.DataFrame(scaled_data, columns=data.columns)


scaler = StandardScaler()
scaled_data_all = scaler.fit_transform(data_all)
scaled_data_all = pd.DataFrame(scaled_data_all, columns=data_all.columns)


In [ ]:
correlation_matrix = scaled_data_all.corr()

plt.figure(figsize=(18, 14))
mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))
sns.heatmap(correlation_matrix, mask=mask, cmap='viridis', center=0, vmin=-1, vmax=1,
            linewidths=0.3, annot=False)
plt.title('Matrice de corrélation des features', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
threshold = 0.80
high_correlation_matrix = []
for i in range(len(correlation_matrix.columns)):
    for j in range(i):
        if correlation_matrix.iloc[i, j] > threshold:
            high_correlation_matrix.append((correlation_matrix.columns[i], correlation_matrix.columns[j], correlation_matrix.iloc[i, j]))

if high_correlation_matrix:
    print(f"Features très semblables (r > {threshold} )")
    for a, b, r in sorted(high_correlation_matrix, key=lambda x: abs(x[2]), reverse=True):
        print(f'  {a} ~ {b}  r={r:.3f}')

In [ ]:
data_all_plot = data_all.copy()
data_all_plot["Target"] = data_all_plot["Target"].map({0: "Dropout", 1: "Graduate/Enrolled"})

for attribute in data.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(
        data=data_all_plot,
        x=attribute,
        hue="Target",
        hue_order=["Dropout", "Graduate/Enrolled"]  # ordre fixe de la légende
    )
    plt.title(f'Distribution de {attribute} par classe')
    plt.xlabel(attribute)
    plt.ylabel('Fréquence')
    plt.tight_layout()
    plt.show()



In [ ]:
# PCA
import matplotlib.pyplot as plt
from sklearn import decomposition

for size in [2, 5, 10]:
    pca = decomposition.PCA(n_components=size)
    pca.fit(scaled_data_all)
    data2 = pca.transform(scaled_data_all)

    plt.scatter(data2[:,0], data2[:,1], c=data_target)
    plt.legend(['Dropout', 'Graduate/Enrolled'])
    plt.title(f'PCA avec {size} composantes')
    plt.show()

In [ ]:
from pandas import DataFrame


pca = decomposition.PCA(n_components=2)
pca.fit(scaled_data_all)
data2 = pca.transform(scaled_data_all)
data2 = pd.DataFrame(data2, columns=[f'PC{i+1}' for i in range(2)])
data3main: DataFrame = data2.loc[data2['PC1'] > -5, :]
data3sub: DataFrame = data2.loc[data2['PC1'] <= -5, :]
target3main = data_target.loc[data3main.index]
target3sub = data_target.loc[data3sub.index]

plt.scatter(data3main['PC1'], data3main['PC2'], c=target3main)
plt.legend(['Dropout', 'Graduate/Enrolled'])
plt.title(f'PCA avec {size} composantes')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.show()


plt.scatter(data3sub['PC1'], data3sub['PC2'], c=target3sub)
plt.legend(['Dropout', 'Graduate/Enrolled'])
plt.title(f'PCA avec {size} composantes')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.show()

## Interprétation des résultats

##### Certains attributs ne semble pas être des facteurs déterminants puisqu'ils ont la même densité pour les dropout que les autres. Ces attributs sont principalement :
-GDP; Inflation Rate;

##### Certains attributs semblent être des facteurs déterminants
- Age (aux alentours de 20 réside toute la concentration de dropout)
- Scholarship (beaucoup plus de dropout chez ceux en possédant 1)
- Gender (plus de dropout chez 1 -> male)
- Displaced (plus de dropout chez 0 -> non displaced)
- Debtor (plus de dropout chez 1 -> debtor)
- Tuition fees (moins de dropout chez ceux qui payent des frais de scolarité)


## Question 3

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(scaled_data, data_target, test_size=0.1, random_state=42)

In [ ]:
X_train

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier

rf = GradientBoostingClassifier()
rf.fit(X_train, y_train)
print(f"GradientBoostingClassifier initial accuracy: train {rf.score(X_train, y_train)} / test {rf.score(X_test, y_test)}")

#### Interprétation

La différence entre le résultat sur les données de train et de test peut venir de plusieurs facteurs. Le premier est les outliers qui peuvent influer sur ces précisions. La seconde raison est l'agorithme choisi qui overfit plus ou moins. Celui que nous avons choisi, GradientBoostingClassifier limite cela en limitant la profondeur des arbres sur lesquels il regarde. Egalement il limite le nombre d'arbres sur lesquels il s'entraine. Par contre, l'algorithme RandomForestClassifier est entièrement déterministe (accuracy de 1 sur les données train), il construit l'arbresur les données qu'il a vu donc il s'en souvient.

## Question 4

In [ ]:
rf = RandomForestClassifier()
rf.fit(X_train, y_train)
print(f"Random Forest initial accuracy (random_state): train {rf.score(X_train, y_train)} / test {rf.score(X_test, y_test)}")

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

rf_accuracies = []
gb_accuracies = []
seed_range = [10, 20, 30, 40, 50, 60, 70, 80, 90]

for seed in seed_range:
    X_train, X_test, y_train, y_test = train_test_split(scaled_data, data_target, test_size=0.1, random_state=seed)
    # Random Forest
    rf_tmp = RandomForestClassifier()
    rf_tmp.fit(X_train, y_train)
    rf_accuracies.append(rf_tmp.score(X_test, y_test))

    # Gradient Boosting
    gb_tmp = GradientBoostingClassifier()
    gb_tmp.fit(X_train, y_train)
    gb_accuracies.append(gb_tmp.score(X_test, y_test))

import matplotlib.pyplot as plt

plt.plot(seed_range, rf_accuracies, label='Random Forest', marker='o')
plt.plot(seed_range, gb_accuracies, label='Gradient Boosting', marker='s')
plt.xlabel('Random State Seed')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

## Interprétation des résultats

Les résultats sont différents car nous avons environ 1/3 de target 0 et 2/3 de target 1.
Random splitting techniques like train_test_split() or regular K-Fold can create problem if they produce imbalanced class proportions in the training and test sets. For example imagine a binary classification dataset with 100 samples where:

    80 samples are Class 0
    20 samples are Class 1

Using random sampling in an 80:20 split then all 80 Class 0 in the training set and all 20 Class 1 in the test set. In this case model will never learn to classify Class 1 and would give misleading accuracy.


Donc en fonction de la seed choisie, les training/test sets peuvent tenir plus ou moins compte de la situation réelle.

## Question 5

In [ ]:
from sklearn.metrics import confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay

gb = GradientBoostingClassifier()
gb.fit(X_train, y_train)
pred = gb.predict(X_test)
cm = confusion_matrix(y_test, pred, normalize='true')
cm = cm
print(cm)

In [ ]:

cmd = ConfusionMatrixDisplay(cm)
cmd.plot()

In [ ]:
# f1-score
f1_score(y_test, pred)

#### Interprétation

| Terme | Définition                                   |
|-------|----------------------------------------------|
| **TP** | Étudiants Non-Dropout correctement prédits Non-Dropout |
| **TN** | Étudiants Dropout correctement prédits Dropout |
| **FP** | Étudiants Dropout mal prédits Non-Dropout    |
| **FN** | Étudiants Non-Dropout mal prédits Dropout    |

Avantage du F1-score sur l'accuracy:
L'accuracy peut être trompeuse sur des classes déséquilibrées (ex : si 80% sont Non-Dropout, un classifieur naïf qui prédit toujours 1 atteint 80% d'accuracy sans rien apprendre).
Le F1-score est la moyenne harmonique de la précision et du rappel, ce qui pénalise les faux positifs et les faux négatifs de façon équilibrée.

## Question 6

In [ ]:

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.99, random_state=seed)

# Gradient Boosting
gb_tmp = GradientBoostingClassifier()
gb_tmp.fit(X_train, y_train)
print("Gradient Boosting Accuracy with 1% training data:", gb_tmp.score(X_test, y_test))

#### Interprétation

Avec si peu de données, le modèle n'a pas assez d'exemples pour apprendre les patterns. Il a un fort variance,  il sur apprend les quelques exemples vus et généralise mal.

On peut améliorer la qualité du classifier avec très peu de données.
Un Decision Tree avec profondeur limitée (`max_depth=4`) a moins de paramètres et donc biais plus fort mais une variance réduite et donc meilleure généralisation dans ce régime de données limitées.

C'est le biais-variance trade-off : quand les données sont rares, préférer des modèles plus simples qui sont moins propices à l'overfitting.

# 2. Regression

## Question 7

Import the necessary libraries

In [ ]:
import pandas as pd
import sklearn as sl
import plotly.express as px
import plotly as pl
import numpy as np
from sklearn.metrics import mean_squared_error

Load the dataset `measurements.csv` in a dataframe

In [ ]:
data = pd.read_csv('measurements.csv')
data.rename(columns={'0': 'x', '1': 'y'}, inplace=True)
data

Plotting the data in a scatter plot

In [ ]:
pl.plot(data, kind='scatter', x='x', y='y')

Fitting the linear regression model

In [ ]:
reg = sl.linear_model.LinearRegression()
x = np.array(data['x'])
y = np.array(data['y'])

reg.fit(x.reshape(-1, 1), y)

line = reg.coef_ * x + reg.intercept_
line_df = pd.DataFrame({'x': x, 'y': line})

Plot the regrsession line and the data points

In [ ]:
fig = px.line(line_df, x='x', y='y')
fig.add_scatter(x=data['x'], y=data['y'], mode='markers', name='Data')
fig.update_layout(title='Linear Regression')
fig.show()

Compute R² and MSE

In [ ]:
sl.metrics.r2_score(data['y'], line_df['y'])

In [ ]:
mean_squared_error(data['y'], line_df['y'])

#### MSE
- L'erreur absolue moyenne est relativement proche de 0 (0.49), ce qui veut dire qu'en moyenne les points sont proches de l'estimation linéaire faite.

#### R²
- Le coefficient de regression linéaire est proche de 1 (0.874), donc c'est un bon modèle

## Question 8

In [ ]:
degrees = [2, 8, 25]

for d in degrees:
    reg = sl.preprocessing.PolynomialFeatures(d, include_bias=False)
    x = np.array(data['x'])
    y = np.array(data['y'])
    features = reg.fit_transform(x.reshape(-1, 1))

    l_reg = sl.linear_model.LinearRegression()
    l_reg.fit(features, y)
    coefficients = l_reg.coef_
    intercept = l_reg.intercept_


    x_range = np.arange(start=x.min() - 2, stop=x.max() + 2, step=1)
    features_range = reg.transform(x_range.reshape(-1, 1))
    y_range = l_reg.predict(features_range)

    line_df = pd.DataFrame({'x': x_range, 'y': y_range})

    fig = px.line(line_df, x='x', y='y')
    fig.add_scatter(x=data['x'], y=data['y'], mode='markers', name='Data')
    fig.update_layout(title=f'Polynomial regression with degree {d}')
    fig.show()

    y_pred = l_reg.predict(features)
    r2 = sl.metrics.r2_score(data['y'], y_pred)
    mse = mean_squared_error(data['y'], y_pred)

    print(f"R2: {r2}")
    print(f'MSE: {mse}')


 Plus le degré est haut, plus ça a tendance à overfitter. Le problème est que quand on a un degré de polynôme supérieur au nombre de points, on obtient un polynôme qui n'a plus vraiment de sens. Donc il faut garder un degré de polynôme toujours inférieur au nombre de points.